#Incremental data loading using Auroloader

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS netflx_catalog.netflix_schema;

In [0]:
checkpoint_location = "abfss://silver@mynetflxstrgacc.dfs.core.windows.net/checkpoint"

In [0]:
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation",checkpoint_location) \
  .load("abfss://raw@mynetflxstrgacc.dfs.core.windows.net/netflix_titles")

In [0]:
try:
    df.columns  # Triggers eager analysis for error detection
    display(df)
except Exception as e:
    print(f"Error: {repr(e)}")

Error: AnalysisException()


In [0]:
df.writeStream.format("delta") \
  .option("checkpointLocation", checkpoint_location) \
  .trigger(processingTime="10 seconds") \
  .start("abfss://bronze@mynetflxstrgacc.dfs.core.windows.net/netflix_titles") 